# BioGen AI: RF Diffusion Colab Backend
This notebook runs a FastAPI server in Google Colab and exposes it to the internet using `pyngrok`.
Your frontend application will connect to the URL generated at the bottom of this notebook.

In [ ]:
!pip install fastapi uvicorn pyngrok nest_asyncio pydantic
# (In a full implementation, you would also install RF Diffusion here)

In [ ]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI, HTTPException, Header
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uuid
import time

app = FastAPI(title="BioGen AI Colab Server")

# Allow CORS from anywhere for testing purposes
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Basic authentication function
def verify_api_key(x_api_key: str = Header(None)):
    # In production, check against a secret key
    # if x_api_key != "YOUR_SECRET_KEY":
    #     raise HTTPException(status_code=401, detail="Unauthorized")
    pass

jobs_db = {}

@app.get("/api/v1/health")
def health_check():
    return {"status": "ok", "gpu_available": True}

class JobSubmit(BaseModel):
    target_protein: str = None
    sequence_length: int = 100
    # Add other RF Diffusion params here

@app.post("/api/v1/jobs")
def submit_job(job: JobSubmit):
    job_id = str(uuid.uuid4())
    jobs_db[job_id] = {
        "id": job_id,
        "status": "processing",
        "params": job.dict(),
        "created_at": time.time()
    }
    
    # Here you would trigger an async task to run the actual RF diffusion CLI.
    # For demonstration, we will simulate it finishing quickly.
    return {"job_id": job_id, "status": "processing"}

@app.get("/api/v1/jobs/{job_id}")
def get_job(job_id: str):
    if job_id not in jobs_db:
        raise HTTPException(status_code=404, detail="Job not found")
    
    job = jobs_db[job_id]
    # Simulate job completion after 10 seconds
    if job["status"] == "processing" and (time.time() - job["created_at"] > 10):
        job["status"] = "completed"
        # In reality, this would contain the URL or path to the downloaded PDB
        job["result_pdb"] = "mock_pdb_data_here"

    return job

class PdbFetch(BaseModel):
    pdb_id: str

@app.post("/api/v1/pdb/fetch")
def fetch_pdb(req: PdbFetch):
    return {"pdb_data": "mock data for " + req.pdb_id}

if __name__ == "__main__":
    import os
    # IMPORTANT: You must sign up at ngrok.com and get your auth token
    ngrok_token = os.environ.get("NGROK_AUTH_TOKEN", "")
    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
    else:
        print("WARNING: NGROK_AUTH_TOKEN not set. Tunnel might expire quickly or fail.")
    
    ngrok_tunnel = ngrok.connect(8000)
    print("\n" + "="*60)
    print("\n\t\tCOLAB PUBLIC URL:\n")
    print(f"\t\t{ngrok_tunnel.public_url}\n")
    print("\t\tCopy and paste this URL into your frontend Settings.\n")
    print("="*60 + "\n")
    
    nest_asyncio.apply()
    uvicorn.run(app, port=8000, host="0.0.0.0")
